# Data analysis, analysis and confirmation（データの分析と解析と確認）

* Check the content and structure of the data, and investigate and study the meaning and role of each feature.
(データの中身を、構造を確認し、それぞれの特徴の意味や役割を調べる、勉強する)

* Github RepoURL→[Enveda_CASMI_2026](https://github.com/kenkenkengo0421/Enveda_CASMI_2026)

In [ ]:
import polars as pl
import os
import sys
from pathlib import Path 
import matplotlib.pyplot as plt
import warnings

try:
    import google.colab
    from google.colab import drive
    is_colab = True
except ImportError:
    is_colab = False

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    print("：Kaggle")
    sys.path.append("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra")
    sys.path.append("/kaggle/input/datasets/nagatakengo/enveda-casmi-2026")
    train_data_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
    test_data_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"
    from function import *
    from plot import *


elif is_colab:
    print("：Colab")
    drive.mount("/content/drive")
    !mkdir -p /content/dataset
    !unzip -q -o "/content/drive/MyDrive/colab/Enveda_CASMI_2026/enveda-CASMI26-molecule-id-mass-spectra.zip" -d /content/dataset/
    train_data_path = "/content/dataset/train.parquet"
    test_data_path = "/content/dataset/test.parquet"
    from function import *
    from plot import *
    
else:
    print("：local PC")
    train_data_path = '../content/train.parquet'
    test_data_path = '../content/test.parquet'
    root_dir = Path(os.getcwd()).parent
    if str(root_dir) not in sys.path:
        sys.path.append(str(root_dir))
    from source.function import *
    from source.plot import *

p_train_df = pl.scan_parquet(train_data_path)
p_test_df = pl.scan_parquet(test_data_path)

# Check train data(学習データの確認)

* train.parquet：18 columns

>*  'ingest_lib',                     : Provider library name (11 types below), 提供元ライブラリ名（下記11種） ※1 
>*  'normalized_smiles',              : Correct chemical structure (standardized SMILES), 正解の化学構造（標準化SMILES）
>*  'inchikey',                       : Complete InChIKey, 完全なInChIKey
>*  'inchikey14',                     : InChIKey 1st block to identify 2D skeleton (evaluation criteria), 2D骨格を識別するInChIKey第1ブロック（評価基準）
>*  'molecular_formula', : Chemical formula of neutral structure, 中性構造の化学式
>*  'ionization_mode', : positive or negative, 正（positive）または負（negative）
>*  'instrument_type', : Name of the measuring device (test is always 'timsTOF'), 測定機器の名称（testは常に 'timsTOF'）
>*  'adduct', : adduct, 付加体
>*  'adduct_orig', : Original adduct string before standardization, 標準化前の元の付加体文字列
>*  'precursor_mz', : Measured precursor ion mass-to-charge ratio (m/z), 測定された前駆体イオンの質量電荷比（m/z）
>*  'precursor_error_ppm', : Measurement error from theoretical value (ppm, indicator of label quality), 理論値との測定誤差（ppm、ラベル品質の指標）
>*  'ms2_mzs', : Array of fragment m/z values(fragment weight), フラグメントm/z値の配列（破片の重さ）
>*  'ms2_normalized_intensities', : Normalized intensity array (maximum peak=1.0), 正規化された強度配列（最大ピーク=1.0）
>*  'num_peaks', : Number of fragment peaks, フラグメントピーク数
>*  'base_peak_intensity', : Raw counts before normalization of maximum peak, 最大ピークの正規化前の生カウント数
>*  'collision_energy_ev', : Collision energy (numeric list in eV conversion), 衝突エネルギー（eV換算の数値リスト）
>*  'collision_energy_orig', : Original collision energy notation recorded on the device, 機器に記録されていた元の衝突エネルギー表記
>*  'collision_energy_orig_units' : Original collision energy units (test is always 'eV'), 元の衝突エネルギー単位（testは常に 'eV'）



## Check 11 types of libraries in 'ingest_lib'('ingest_lib'の11種類のライブラリを確認)


 ※1

* enveda-180: 1,153,785 spectra / 182,941 structures (timsTOF. Same instrument as the test set, but with a different chemical space due to being from a synthetic small molecule drug discovery screening system)
* pluskal_ms2: 527,581 spectra / 46,821 structures (Orbitrap)
* riken: 347,171 spectra / 15,892 structures (centered on plant metabolites)
* gnps: 220,849 spectra / 45,750 structures (largest natural product reference, heterogeneous due to community submissions)
* massbank: 101,727 spectra / 9,180 structures
* mona: 92,416 spectra / 11,681 structures
* spectraverse: 50,933 spectra / 9,631 structures
* msdial: 40,765 spectra / 9,127 structures
* drug_plus: 2,545 spectra / 2,539 structures
* enveda-np-examples: 1,151 spectra / 250 structures (250 types of natural products generated using the same timsTOF instrument and the same processing pipeline as the test set)
* masaryk: 652 spectra / 416 structures

＝＝和訳

* enveda-180: 1,153,785スペクトル / 182,941構造（timsTOF。テストと同機器だが合成低分子創薬スクリーニング系で化学空間が異なる）   
* pluskal_ms2: 527,581スペクトル / 46,821構造（Orbitrap）  
* riken: 347,171スペクトル / 15,892構造（植物代謝物中心）  
* gnps: 220,849スペクトル / 45,750構造（天然物リファレンス最大、コミュニティ投稿で不均一）   
* massbank: 101,727スペクトル / 9,180構造   
* mona: 92,416スペクトル / 11,681構造   
* spectraverse: 50,933スペクトル / 9,631構造   
* msdial: 40,765スペクトル / 9,127構造   
* drug_plus: 2,545スペクトル / 2,539構造   
* enveda-np-examples: 1,151スペクトル / 250構造（テストセットと同じtimsTOFかつ同じ処理パイプラインで作られた天然物250種）   
* masaryk: 652スペクトル / 416構造





In [ ]:
first_check(p_train_df, "train.parquet")

# Check test data(テストデータの確認)

* test.parquet： 12columns

>* 'molecule_id', : Molecule identifier (submission key)分子の識別子（提出のキー)
>* 'spectrum_id', : Identifier for each spectrum(各スペクトルの識別子)
>* 'ms2_mzs', : Array of fragment m/z values(fragment weight)フラグメントm/z値の配列（破片の重さ）
>* 'ms2_normalized_intensities',: Normalized intensity array (maximum peak=1.0)正規化された強度配列（最大ピーク=1.0）
>* 'base_peak_intensity', : Raw counts before normalization of maximum peak最大ピークの正規化前の生カウント数
>* 'adduct',: Adducts (testing limited to 10 types: [M+H]+, [M+NH4]+, [M-H2O+H]+, [M-2H2O+H]+, [M+Na]+, [M+K]+, [M-H]-, [M-H2O-H]-, [M+CH2O2-H]-, [M+Cl]-
付加体（テストは [M+H]+, [M+NH4]+, [M-H2O+H]+, [M-2H2O+H]+, [M+Na]+, [M+K]+, [M-H]-, [M-H2O-H]-, [M+CH2O2-H]-,[M+Cl]- の10種に限定）
>* 'ionization_mode', : positive or negative正（positive）または負（negative）
>* 'instrument_type', : Name of measuring equipment測定機器の名称（testは常に 'timsTOF'）
>* 'precursor_mz', : Measured precursor ion mass-to-charge ratio (m/z)測定された前駆体イオンの質量電荷比（m/z）
>* 'collision_energy_orig', : Original collision energy notation recorded on the device
機器に記録されていた元の衝突エネルギー表記
>* 'collision_energy_ev', : Collision energy (numeric list in eV conversion)衝突エネルギー（eV換算の数値リスト）
>* 'collision_energy_orig_units' : Original collision energy units (test is always 'eV')元の衝突エネルギー単位（testは常に 'eV'）

In [ ]:
first_check(p_test_df, "test.parquet")

#  ingest_lib
* What is a “library”?
A database (spectrum library) that records and publishes previously measured mass spectrometry spectra and the correct structure of the compound as a set.

(「ライブラリ」とは?過去に測定された質量分析スペクトルと、その化合物の正解構造をセットで記録・公開しているデータベース（スペクトルライブラリ）のこと)

* Approximately 2.5 million spectra in the training data (train.parquet) are imported (collected) from 11 different public databases around the world, and “that one row of spectra
The ingest_lib column records what database the file was brought from.

(学習データ（train.parquet）の約250万件のスペクトルは、世界中の異なる11の公開データベースから取り込まれて（集められて）おり、「その1行のスペクトル)

* The material clearly states that it was collected from the following 11 organizations/databases:

(資料には以下の11組織・データベースから収集されたと明記されています)




|name|explanation|
|----|----|
|enveda-180|Data centered on synthetic compounds measured by organizer Enveda<br>主催者Enveda社が測定した合成化合物中心のデータ|
|gnps|Largest natural products community data<br>最大規模の天然物コミュニティデータ|
|riken|Data centered on plant metabolites published by RIKEN in Japan<br>日本の理化学研究所が公開している植物代謝物中心のデータ|
|pluskal_ms2|Data measured with a device called Orbitrap<br>Orbitrapという機器で測定されたデータ|
|massbank<br>mona<br>msdial<br>spectraverse|Spectrum DB published by each research institution/project<br>各研究機関・プロジェクトが公開しているスペクトルDB|
|enveda-np-examples|Natural product data (250 compounds) created in the same measurement environment as the test<br>テストと同じ測定環境で作られた天然物データ（250化合物）|
|drug_plus<br>masaryk|Small-scale data such as pharmaceuticals and standard materials<br>医薬品や標準物質などの小規模データ|
 

In [ ]:
columns_summary(p_train_df, 'ingest_lib')

# inchikey

* `inchikey`: A 27-character fixed-length alphabetic hash code developed to uniquely identify chemical structures. It also includes information such as stereochemistry (3D orientation) and charge.

* `inchikey14`: This is the hash value of the first block (first 14 characters) of the complete InChIKey. It only represents the "two-dimensional skeleton (bonding relationships between atoms)" excluding the three-dimensional orientation and tautomers.

* String structure
````
It consists of three blocks separated by hyphens like "XXXXXXXXXXXXXX-YYYYYYYYYY-Z".
1st block (first 14 characters: inchikey14):
This is a hash value that represents the "two-dimensional skeleton (bond topology of which atoms are bonded to which atoms)" of the molecule.
Differences in steric orientation (front/back) and tautomers (isomerization due to hydrogen movement) are not reflected.

Second block (10 characters):
Represents steric configuration (3D orientation), isotope information, charge, etc.

Third block (1 character):
This code represents the version of InChI used.
````
* In the SMILES notation, which expresses chemical structures as strings, the string changes depending on which atom you start writing from, even for the same molecule.
InChIKey is a hash value, so if it has the same structure, it will always be the exact same string
* Since the evaluation of this competition does not consider stereoscopic information (3D), correctness will be determined only by whether the first 14 characters (inchikey14) after the second block are the same.
* GroupKFold group key: In order to prevent the same compound from leaking between training data and validation data, it is specified and recommended to use this inchikey14 instead of spectrum ID or row unit for the group unit of division.


* `inchikey`: 化学構造を一意に識別するために開発された27文字の固定長英字ハッシュコードです。立体化学（3Dの向き）や電荷などの情報も含んでいます。

* `inchikey14`: 完全なInChIKeyの第1ブロック（先頭14文字）を切り出したハッシュ値です。立体の向きやトートマーを除いた「2次元の骨格（原子同士の結合関係）」のみを表します。

* 文字列の構造
```
「XXXXXXXXXXXXXX-YYYYYYYYYY-Z」のようにハイフン区切りの3つのブロックで構成されています。
第1ブロック（先頭14文字：inchikey14）:
分子の「2次元骨格（どの原子とどの原子が結合しているかという結合トポロジー）」を表すハッシュ値です。
立体的な向き（手前・奥）やトートマー（水素の移動による異性化）の違いは反映されません。

第2ブロック（10文字）:
立体配置（3Dの向き）や同位体情報、電荷などを表します。

第3ブロック（1文字）:
使用したInChIのバージョン等を表すコードです。
```


* 化学構造を文字列で表すSMILES記法は、同じ分子であっても「どの原子から書き始めるか」によって文字列が変わってしまいます。
InChIKeyはハッシュ値であるため、同じ構造であれば必ず完全に同一の文字列になります
* 本コンペの評価では立体情報（3D）を問わないため、第2ブロック以降を切り捨てた先頭14文字（inchikey14）が一致しているかどうかだけで正誤を判定します。
* GroupKFoldのグループキー: 学習データと検証データの間で同じ化合物が漏洩（リーク）することを防ぐため、分割のグループ単位にはスペクトルIDや行単位ではなく、この inchikey14 を使用することが指定・推奨されています。

In [ ]:
columns_summary(p_train_df, 'inchikey', 10)
columns_summary(p_train_df, 'inchikey14', 10)

# normalized_smiles

* `normalized_smiles`:
String data representing the "correct chemical structure" corresponding to each spectrum (exists only in train.parquet).
* SMILES notation: A standard format in which the molecular structure (types of atoms and bond connections) is expressed as a one-line string using alphabets and symbols (C, O, =, numbers representing rings, etc.)
Meaning of "normalized": SMILES has a property that the character string changes depending on the atom at which it is written, so it indicates a state in which the notation variations are standardized (canonicalization) according to certain rules.



* `normalized_smiles` :
各スペクトルに対応する「正解の化学構造」を表す文字列データです（train.parquet のみに存在）。
* SMILES記法: 分子構造（原子の種類や結合のつながり）を、アルファベットや記号（C, O, =, 環を表す数字など）で1行の文字列として表現した標準的な形式
「normalized（標準化）」の意味: SMILESは書き始めの原子によって文字列が変わる性質があるため、一定のルールで表記揺れを統一（Canonicalization）した状態であることを示しています。





In [ ]:
columns_summary(p_train_df, 'normalized_smiles', 10)

# molecular_formula
* `molecular_formula`: The neutral molecular formula (formula) of the correct compound in each spectrum (train.(Only available in parquet).
Stored value: A string representing the element symbol and the number of atoms (e.g. C18H19BrN2O2).
The meaning of Neutral: It represents the chemical formula as the original molecule itself, which does not contain the adduct (ion) attached at the time of measurement.
* test.It is not included in parquet.
* From the measured pre-breaking weight (precursor_mz) and adduct (adduct), it is possible to reverse and annotate with high accuracy using tools (SIRIUS, MIST・CF, etc.).
* Due to the high accuracy of mass spectrometry, it is relatively easy to calculate the molecular formula from the precursor m/z itself with an accuracy of 95% or more
* Even if the molecular formula (material) is determined to be 1, "compounds (structural isomers) that have exactly the same molecular formula but have different ways of connecting atoms and shapes" will hit hundreds or thousands of kinds in the database.
* The essence of this competition is not "to guess the molecular formula", but "to select the structure of the correct answer from a huge number of candidates with the same molecular formula, using the debris pattern (spectrum) as a clue to the 1st place".
* "Annotation" in this competition refers to "assigning information (labels and annotations) to the measurement data as what it is."


* `molecular_formula` :各スペクトルの正解化合物における「中性分子式（化学式）」です（train.parquet のみに存在）。
格納されている値: 元素記号と原子の個数を表す文字列です（例: C18H19BrN2O2）。
中性（Neutral）の意味: 測定時にくっついた付加体（イオン）を含まない、元の分子単体としての化学式を表しています。
* test.parquet には含まれていません。
* 測定された壊す前の重さ（precursor_mz）と付加体（adduct）から、ツール（SIRIUSやMIST-CFなど）を用いて高精度に逆算・アノテーションすることが可能です。
* 質量分析の精度が高いため、前駆体m/zから分子式を当てること自体は95%以上の精度で比較的簡単に達成できます
* 分子式（材料）が1つに定まっても、「全く同じ分子式なのに原子のつながり方や形が違う化合物（構造異性体）」がデータベース内に何百〜何千種類もヒットします。
* このコンペの本質は「分子式を当てること」ではなく、「同じ分子式を持つ膨大な候補の中から、破片パターン（スペクトル）を手がかりに正解の構造を1位に選び出すこと」にあります
* このコンペにおける「アノテーション（Annotation）」とは、「測定データに対して、それが何であるかという情報（ラベルや注釈）を割り当てること」を指します。

In [ ]:
columns_summary(p_train_df, 'molecular_formula', 10)

# ionization_mode
* `ionization_mode` : An experimental condition that indicates the "polarity (orientation)" when the molecule is charged (ionized) when the molecule is measured with a mass spectrometer (common column for train / test).
Stored values: These are 2 types of strings.

* positive (positive ion mode) negative (negative ion mode)

* Linkage with adduct (adduct):
For positive: Positive charged ions such as [M+H]+, [M+Na]+, and [M+NH4]+ stick together
For negative: [M-H]-, [M+Cl]-, It produces negatively charged ions such as [M+CH2O2-H]-.
* Difference in collision_energy notation:
Some measuring instruments record a negative value (e.g. -40) in collision_energy_orig when in negative mode.
* However, the converted Because collision_energy_ev is uniform to positive values, it is recommended to refer to collision_energy_ev in preprocessing
* When performing similarity search and feature creation of spectra, it is basic to compare and learn by matching the modes (positive / negative).


* `ionization_mode` :質量分析計で分子を測定する際、分子に電気を帯びさせた（イオン化させた）ときの「極性（向き）」を示す実験条件です（train / test 共通カラム）。
格納されている値: 以下の2種類の文字列です。

* positive（正イオンモード）negative（負イオンモード）

* adduct（付加体）との連動:
positive の場合: [M+H]+, [M+Na]+, [M+NH4]+ などの正電荷イオンがくっつきます
negative の場合: [M-H]-, [M+Cl]-, [M+CH2O2-H]- などの負電荷イオンが生じます。
* collision_energy（衝突エネルギー）の表記差:
一部の測定機器では、negative モードのときに collision_energy_orig に負値（例: -40）が記録されています。
* ただし、換算済みの collision_energy_ev では正値に統一されているため、前処理では collision_energy_ev を参照することが推奨されます
* スペクトルの類似度検索や特徴量作成を行う際は、モード（positive / negative）を一致させて比較・学習させるのが基本となります。

In [ ]:
columns_summary(p_train_df, 'ionization_mode', 2)

# instrument_type
* `instrument_type`: This column shows the name and type of the instrument (instrument) that measured the mass spectrometry spectrum (common to train / test).
* test.parquet: All are unified in timsTOF (Bruker timsTOF).
* train.parquet: Multiple devices are mixed because of the wide variety of source libraries.

Examples: enveda-180 and enveda-np-examples are timsTOF, but pluskal_ms2 is measured with Orbitrap.
* Bruker (Bruker): It is the name of a leading precision analysis equipment manufacturer based in Germany.
* timsTOF: The name of a series of mass spectrometers developed and manufactured by the company.
* TIMS (Trapped Ion Mobility Spectrometry): A technology that separates ions in a gas by "shape and size (mobility)".
* TOF (Time-of-Flight / Time-of-Flight Type): Technology to measure "weight (m/z)" with high accuracy from the speed at which ions fly.

* `instrument_type`: 質量分析スペクトルを測定した「装置（機器）の名称・種類」を示すカラムです（train / test 共通）。
* test.parquet: すべて timsTOF（Bruker timsTOF）に統一されています。
* train.parquet:提供元ライブラリが多岐にわたるため、複数の機器が混在しています。

例: enveda-180 や enveda-np-examples は timsTOF ですが、pluskal_ms2 は Orbitrap で測定されています。
* Bruker（ブルカー）: ドイツに本拠を置く大手精密分析機器メーカーの名称です。
* timsTOF（ティムストフ）: 同社が開発・製造している質量分析装置のシリーズ名です。
* TIMS（Trapped Ion Mobility Spectrometry）: 気体中のイオンの「形や大きさ（移動度）」で分ける技術。
* TOF（Time-of-Flight / 飛行時間型）: イオンが飛ぶスピードから「重さ（m/z）」を高精度に測定する技術。

In [ ]:
columns_summary(p_train_df, 'instrument_type', 10)

In [ ]:
columns_summary(p_test_df, 'instrument_type', 10)

# adduct
* `adduct`: A column (common to train / test) that shows the ion species bound (or desorbed) to the molecule when ionizing the molecule with a mass spectrometer.
* Physical role: adduct from measured weight (precursor_mz) It is used to reverse the original exact molecular weight (neutral mass) before breaking (e.g. [M+H]+ subtracts the proton mass of about 1.0073 Da).
* test.parquet: Limited to the following 10 types.
Positive ions: [M+H]+, [M+NH4]+, [M-H2O+H]+, [M-2H2O+H]+, [M+Na]+, [M+K]+
Negative ions: [M-H]-, [M-H2O-H]-, [M+CH2O2-H]-, [M+Cl]-
* train.parquet:
Many adducts other than the above 10 species are also mixed.
The original notation before the standardization was recorded in train. The adduct_orig column also exists.


* `adduct`: 質量分析計で分子をイオン化する際に、分子に結合（または脱離）したイオン種を示すカラム（train / test 共通）。
* 物理的な役割: 測定された重さ（precursor_mz）から adduct の質量を差し引くことで、壊す前の元の正確な分子量（中性質量）を逆算するために使用します（例: [M+H]+ ならプロトン質量 約1.0073 Da を引く）。
* test.parquet: 以下の10種類に限定されています。
正イオン: [M+H]+, [M+NH4]+, [M-H2O+H]+, [M-2H2O+H]+, [M+Na]+, [M+K]+
負イオン: [M-H]-, [M-H2O-H]-, [M+CH2O2-H]-, [M+Cl]-
* train.parquet:
上記10種以外の付加体も多数混在しています。
train には標準化前の元表記を記録した adduct_orig カラムも存在します。

In [ ]:
columns_summary(p_train_df, 'adduct', 10)

In [ ]:
columns_summary(p_test_df, 'adduct', 10)

# adduct_orig
* `adduct_orig` : This is a column that keeps the "original appendage before standardization" that was originally recorded in the data source (each library) as it is (train.(Only available in parquet).
* Differences in data: test.It does not exist in parquet and does not exist in train It will be a unique additional metadata (8 out of 1 column).
* Since it collects data from 11 different public libraries around the world, the way to write in the original data (whether there is space or not, notation rules, etc.) was different even if it was the same additive.
* adduct is the one that the organizer unified and cleansed the notation. The adduct_orig column is the original raw record of the adduct_orig column.

* `adduct_orig` : データ提供元（各ライブラリ）に元々記録されていた「標準化前の元の付加体表記」をそのまま保持しているカラムです（train.parquet のみに存在）。
* データ内の差異: test.parquet には存在せず、train 固有の追加メタデータ（全8列のうちの1つ）となります。
* 世界中の異なる11の公開ライブラリからデータを集めているため、同じ付加体であっても元データでの書き方（スペースの有無や表記ルールなど）がバラバラでした。
* 主催者側で表記を統一・クレンジングしたものが adduct 列であり、元々の生の記録を残したものが adduct_orig 列です。


In [ ]:
columns_summary(p_train_df, 'adduct_orig', 10)

# precursor_mz
* `precursor_mz`: The mass-to-charge ratio (mm/zン) of the precursor ion before breaking (fragmentation) the molecule in the measuring device (common to train/ test).
* Word breakdown: precursor: Ion before breaking m/z: Mass (mmン) divided by charge (ンzン) Data type: Float (test data has a mass range of 157-1,159 Da).
* By subtracting the mass of adduct (adduct) from precursor_mz, the original mass (neutral mass) of the molecule before breaking is calculated.
Example: If the adduct is [M+H]+, we subtract the mass of the hydrogen ion (approximately 1.0073 Da).
* Because precursor_mz has high measurement accuracy, the identification of the molecular formula (material) itself can be achieved with more than 95% accuracy.
However, subsequent analysis of debris data (ms2_mzs, ms2_normalized_intensities) is essential in order to narrow down the correct answer from countless "structural isomers" with the same weight

* `precursor_mz` : 測定装置内で分子を破壊（フラグメンテーション）する前の「前駆体イオンの質量電荷比（$m/z$）」です（train / test 共通）。
* 単語の内訳:precursor（前駆体）: 壊す前のイオンm/z: 質量（$m$）を電荷（$z$）で割った値データ型: Float（テストデータの質量範囲は 157〜1,159 Da）。
* precursor_mz から adduct（付加体）の質量を差し引くことで、壊す前の分子本来の質量（中性質量）を算出します。
例: 付加体が [M+H]+ の場合、水素イオンの質量（約1.0073 Da）を引きます。
* precursor_mz の測定精度が高いため、分子式（材料）自体の特定は95%以上の精度で達成できます。
ただし、同じ重さを持つ無数の「構造異性体」から正解を絞り込むためには、後続の破片データ（ms2_mzs, ms2_normalized_intensities）の解析が不可欠となります

In [ ]:
columns_summary(p_train_df, 'precursor_mz', 10)

In [ ]:
columns_summary(p_test_df, 'precursor_mz', 10)

# precursor_error_ppm
* `precursor_error_ppm`: A column representing the measurement error (deviation) between the weight of the measured precursor ion (precursor_mz) and the "theoretically accurate mass" calculated from the structure of the correct compound (train.(Only available in parquet).
* parts-per-million (relative error of 100 units of 1 million).
* Due to the high accuracy of the mass spectrometer, this error usually falls between a few ppm and a dozen ppm.
* This extremely large error line indicates "calibration deviation of measurement equipment" or "possibility of incorrect registered correct compound label (SMILES) (false annotation)".
* When you remove noise (unreliable data) from the training data, you can use it as a filter condition to exclude samples that have this value too large from the training.


* `precursor_error_ppm` : 測定された前駆体イオンの重さ（precursor_mz）と、正解化合物の構造から計算される「理論上の正確な質量」との測定誤差（ズレ）を表すカラムです（train.parquet のみに存在）。
* parts-per-million（100万分の1単位の相対誤差）。
* 質量分析計は高精度なため、通常この誤差は数 ppm〜十数 ppm 程度に収まります。
* この誤差が極端に大きい行は、「測定機器の校正ズレ」や「登録されている正解化合物ラベル（SMILES）が間違っている可能性（誤アノテーション）」を示唆します。
* 学習データの中からノイズ（信頼性の低いデータ）を取り除く際、この値が大きすぎるサンプルを学習から除外するフィルター条件として活用できます。

In [ ]:
columns_summary(p_train_df, 'precursor_error_ppm', 10)

# ms2_mzs
* `ms2_mzs`: List (sequence) of mass-to-charge ratio (mm/zン) of debris ions (fragment ions) generated when molecules are destroyed (fragmented) in the measuring device (common to train/ test)
* 1:1 support for ms2_normalized_intensities:
ms2_mzs For each fragment weight, the strength of how much that fragment was detected is stored in the same index of ms2_normalized_intensities.
* num_peaks: The number of elements in this list (number of peaks) corresponds to the num_peaks column in train
* Since weak peaks account for more than 9% of the total test data, they are subject to a filtering process that removes peaks with a constant intensity (e.g., less than 1% relative intensity).
* Calculation of neutral loss: By subtracting the weight of each debris (ms2_mzs) from the weight before breaking (precursor_mz), we can identify the chemical groups that have been separated (water -18 Da, CO Da -44 Da, etc.).



* `ms2_mzs`: 測定装置内で分子を破壊（フラグメンテーション）した際に生じた「破片イオン（フラグメントイオン）の質量電荷比（$m/z$）のリスト（配列）」です（train / test 共通）
* ms2_normalized_intensities との1対1対応:
ms2_mzs の各破片の重さに対して、その破片がどれだけ検出されたかという強度が ms2_normalized_intensities の同じインデックスに格納されています。
* num_peaks: このリストに含まれる要素数（ピークの個数）が、train の num_peaks 列に相当します
* テストデータでは微弱なピークが全体の9割以上を占めるため、対応する強度が一定基準（例: 相対強度1%未満）のピークを除去するフィルタリング処理の対象となります。
* ニュートラルロスの算出: 壊す前の重さ（precursor_mz）から各破片の重さ（ms2_mzs）を引くことで、脱離した化学グループ（水 -18 Da、CO₂ -44 Da など）を特定できます

In [ ]:
columns_summary(p_train_df, 'ms2_mzs', 10)

In [ ]:
columns_summary(p_test_df, 'ms2_mzs', 10)

# ms2_normalized_intensities
* `ms2_normalized_intensities`: List (array) of the "strength (detection amount)" of each detected chip ion (fragment ion) (common to train/ test).
* Normalized criteria: The value of the highest peak (base peak) in the spectrum is "1.0", and all intensities are aligned to a relative ratio of 0.0 〜 1.0.
* It is fully linked to the ms2_mzs (debris weight) order, and the strength of each debris is recorded at the same index position
* The maximum peak here is 1.0 However, the actual measured raw counts before they are normalized are kept in the base_peak_intensity column.
* Noise rejection threshold (excluding relative intensity less than 1%): Weak peaks of less than 1% (value less than 0.01) of the peak of the test spectrum account for 93% of the total equipment noise.
* You can truncate peaks that have a value of less than 0.01 in this column to compress them to critical peaks (median 18 or so).
* Intensity transformation (scaling): To mitigate the effects of protruding peaks, it is common to preprocess values with square-root or log transformations.

* `ms2_normalized_intensities` : 検出された各破片イオン（フラグメントイオン）の「強さ（検出量）のリスト（配列）」です（train / test 共通）。
* 正規化（normalized）の基準: そのスペクトル内で最も高いピーク（ベースピーク）の値を「1.0」として、すべての強度が 0.0 〜 1.0 の相対比率に揃えられています。
* ms2_mzs（破片の重さ）の並び順と完全に連動しており、同じインデックスの位置に各破片の強度が記録されています
* ここでは最大ピークが 1.0 に統一されていますが、正規化される前の実際の測定生カウント数は base_peak_intensity 列に保持されています。
* ノイズ除去の閾値（相対強度1%未満の除外）: テストスペクトルのピークのうち、最大ピークの1%未満（値が 0.01 未満）の微弱ピークが全体の93%を占める機器ノイズです。
* この列の値が 0.01 未満のピークを切り捨てることで、重要なピーク（中央値18個程度）に圧縮できます。
* 強度変換（スケーリング）: 突出したピークの影響を緩和するため、値に対して平方根（Square-root）や対数（log）変換を施す前処理が一般的です。

In [ ]:
columns_summary(p_train_df, 'ms2_normalized_intensities', 10)

In [ ]:
columns_summary(p_test_df, 'ms2_normalized_intensities', 10)

# num_peaks
* `num_peaks`: A column representing the total number of peaks of debris ions (fragments) in the measured spectrum (train.(Only available in parquet).
* Relation to other columns: ms2_mzs which is an array (or Matches the number of elements (the length of the list) in (ms2_normalized_intensities
* train.parquet: Stored as a column.
* test.parquet: Does not exist as a column, but ms2_mzs You can get the same value by counting the array length (len(ms2_mzs)).
* According to the data, the median number of peaks in the test spectrum is 230, but weak peaks less than 1% of the maximum peak account for 93% of the total.
* By pre・processing to cut relative intensities less than 1%, compression and noise reduction can be performed to major peaks (median 18 or so).
* It is an index to grasp the difference in the quality and characteristics of data, such as how finely each source library (ingest_lib) picks up the peak, or whether noise removal is performed beforehand.

* `num_peaks` : 測定されたスペクトルに含まれる「破片イオン（フラグメント）のピークの総数」を表すカラムです（train.parquet のみに存在）。
* 他カラムとの関係: 配列である ms2_mzs（または ms2_normalized_intensities）の要素数（リストの長さ）と一致します
* train.parquet: カラムとして保持されています。
* test.parquet: カラムとしては存在しませんが、ms2_mzs の配列長（len(ms2_mzs)）をカウントすることで同一の値を取得できます。
* 資料によると、テストスペクトルのピーク数は中央値230個ですが、最大ピークの1%未満の微弱ピークが全体の93%を占めています。
* 相対強度1%未満をカットする前処理を行うことで、主要なピーク（中央値18個程度）に圧縮・ノイズ除去できます。
* 各提供元ライブラリ（ingest_lib）がどの程度細かくピークを拾っているか、または事前にノイズ除去を施しているかといったデータの品質・特性の差を把握する指標になります。

In [ ]:
columns_summary(p_train_df, 'num_peaks', 10)

# base_peak_intensity
* `base_peak_intensity`: A column representing the "original raw count (raw intensity)" of the highest peak (base peak) in the measured spectrum before being normalized (common to train / test).
* Reference relationship with ms2_normalized_intensities:
* ms2_normalized_intensities records relative intensities with a maximum peak of 1.0.
* This base_peak_intensity holds the absolute intensity value measured by the actual detector before it is normalized to its "1.0".
* A spectrum with an extremely small value can be judged to be a spectrum with a weak detected signal itself and high noise (low quality).

* `base_peak_intensity` : 測定されたスペクトルの中で最も高いピーク（ベースピーク）の、正規化される前の「元の生カウント数（生の強度）」を表すカラムです（train / test 共通）。
* ms2_normalized_intensities との基準関係:
* ms2_normalized_intensities では、最大ピークを 1.0 に揃えた相対強度として記録されています。
* その「1.0」に正規化される前の、実際の検出器で測定された絶対的な強度の値がこの base_peak_intensity に保持されています。
* この値が極端に小さいスペクトルは、検出されたシグナル自体が微弱であり、ノイズが多く含まれている（品質が低い）スペクトルであると判断できます。


In [ ]:
columns_summary(p_train_df, 'base_peak_intensity', 10)

In [ ]:
columns_summary(p_test_df, 'base_peak_intensity', 10)

# collision_energy_ev
* `collision_energy_ev` : This is a numerical list that converts and unifies the "Collision energy" added when destroying molecules in the measuring device into electron volts (eV) units (common to train / test).
* Changes in the way it breaks:
* It shows how strong the force (energy) collided and destroyed the molecule.
* The larger the energy value, the more molecules are destroyed, resulting in more finer and smaller debris peaks (peaks at lowmm/zル).
* Multiple spectra of the same molecule:
* There are several energy-varying spectra of 1 molecule (median 3), and the difference in how each energy breaks is an important clue to structure-specific.
* The raw notation (collision_energy_orig) originally recorded on the instrument has a negative value (-40) in negative ion mode. If you have any questions, please feel free to contact us and we will be happy to answer any questions you may have.
* On the other hand, this collision_energy_ev has been normalized to positive eV units, so it is recommended to always use this for feature creation and preprocessing.

* `collision_energy_ev` : 測定装置内で分子を破壊する際に加えた「衝突エネルギー（Collision Energy）」を、電子ボルト（eV）単位に換算・統一した数値リストです（train / test 共通）
* 壊れ方の変化:
* どれだけ強い力（エネルギー）で分子を衝突・破壊したかを示します。
* エネルギー値が大きいほど分子は激しく破壊され、より細かく小さい破片ピーク（低 $m/z$ のピーク）が多く生じます。
* 同一分子での複数スペクトル:
* 1つの分子に対してエネルギーを変えたスペクトルが複数（中央値3件）測定されており、エネルギーごとの割れ方の違いが構造特定の重要な手がかりになります。
* 機器に元々記録されていた生表記（collision_energy_orig）には、負イオンモードで負値（-40 など）が入っていたり、単位（NCE や V）が混在しています。
* 一方で、この collision_energy_ev は正値の eV 単位に正規化済みであるため、特徴量作成や前処理では必ずこちらを使用することが推奨されています。

In [ ]:
columns_summary(p_train_df, 'collision_energy_ev', 10)

In [ ]:
columns_summary(p_test_df, 'collision_energy_ev', 10)

# collision_energy_orig
* `collision_energy_orig`: This column represents the "original collision energy setting value (raw data)" recorded in the measurement equipment (device) (common to train / test).
* Variation in notation: Because the device records are stored as strings, they contain not only a single value but also multiple step values (e.g. '40', '[20 40 60]').
* Description in negative ion mode:
In negative ion mode (negative) measurement data, a negative value (e.g. '-40') may be recorded as the original setting value.
* Since character strings and negative values ​​are mixed, it is recommended to use the collision_energy_ev column, which has been normalized and digitized into a positive value electron volt (eV) array, for feature creation and analysis.

* `collision_energy_orig` : 測定機器（装置）に記録されていた「元の衝突エネルギーの設定値（生データ）」を表すカラムです（train / test 共通）。
* 表記のばらつき:機器の記録がそのまま文字列として保持されているため、単一の値だけでなく複数のステップ値（例: '40', '[20 40 60]'）などが混在しています。
* 負イオンモードでの表記:
負イオンモード（negative）の測定データでは、元の設定値として負の値（例: '-40'）が記録されている場合があります。
* 文字列かつ負値などが混在しているため、特徴量作成や解析には、正値の電子ボルト（eV）配列に正規化・数値化された collision_energy_ev カラムを使用することが推奨されています。

In [ ]:
columns_summary(p_train_df, 'collision_energy_orig', 10)

In [ ]:
columns_summary(p_test_df, 'collision_energy_orig', 10)

# collision_energy_orig_units
* `collision_energy_orig_units`: A column that represents the "units" of the original collision energy (collision_energy_orig) recorded in the measurement device (common to train / test).
* test.parquet: All are standardized to 'eV' (electron volt).
* train.parquet: Collected from 11 different public libraries, so different units such as 'eV' as well as 'NCE' (normalized collision energy) and 'V' (volts) are mixed.
* Units are mixed in train, so if you use raw numbers (collision_energy_orig) and units (collision_energy_orig_units) as they are for comparison or input, the scales will not match.
* The array column collision_energy_ev was converted and normalized to the positive electron volt (eV) unit by the organizer, taking into account the difference in units.
* Therefore, it is recommended to refer to collision_energy_ev instead of this unit column for machine learning model features and preprocessing.


* `collision_energy_orig_units` : 測定機器に記録されていた元の衝突エネルギー（collision_energy_orig）の「単位」を表すカラムです（train / test 共通）。
* test.parquet: すべて 'eV'（電子ボルト）に統一されています。
* train.parquet: 異なる11の公開ライブラリから集められているため、'eV' だけでなく 'NCE'（規格化衝突エネルギー）や 'V'（ボルト）などの異なる単位が混在しています。
* train 内で単位が混在しているため、生の数値（collision_energy_orig）と単位（collision_energy_orig_units）をそのまま比較・入力として使うとスケールが揃いません。
* 主催者側で単位の違いを考慮し、正値の電子ボルト（eV）単位に換算・正規化してくれた配列カラムが collision_energy_ev です。
* そのため、機械学習モデルの特徴量や前処理には、この単位カラムではなく collision_energy_ev を参照することが推奨されています

In [ ]:
columns_summary(p_train_df, 'collision_energy_orig_units', 10)

In [ ]:
columns_summary(p_test_df, 'collision_energy_orig_units', 10)

# Various identifiers(各識別子)

* molecule_id (String) 分子の一意識別子です。提出ファイル（submission.csv）のキーであり、評価（MRR@25）の集約単位となります。
* spectrum_id (String) 各測定スペクトルの一意識別子です。
* この2つを含めると、コンペティションで配布されている全データ（train 18列 / test 12列）の全カラムがすべて網羅されます。

# Objective variable and explanatory variable (目的変数と説明変数)
1. Objective variable (correct label/prediction target)
* What to predict/submit: SMILES string for each molecule_id (up to 25 in order of confidence)
* Criteria for scoring and determining correct answers: inchikey14 (judged by matching 2D skeletons excluding 3D)
* Correct labels included in `train.parquet`:
* `normalized_smiles` (correct answer structure string)
* `inchikey14` (2D skeletal hash/evaluation key of the correct answer)
* `inchikey` (complete InChIKey of correct answer)
* `molecular_formula` (correct neutral molecular formula)

2. Explanatory variables (features that serve as model inputs)
* Measured value before destruction: `precursor_mz`
* Fragment data: `ms2_mzs`, `ms2_normalized_intensities`, `base_peak_intensity`
* Experimental/measurement conditions: `adduct`, `collision_energy_ev`, `ionization_mode`, `instrument_type`

3. Others (for metadata/data management)
* ID: `molecule_id`, `spectrum_id` (test only)
* Source/quality/original notation: `ingest_lib`, `precursor_error_ppm`, `adduct_orig`, `collision_energy_orig`, `collision_energy_orig_units`, `num_peaks` (train only, etc.)



1. 目的変数（正解ラベル・予測対象）
* 予測・提出するもの: 各 molecule_id に対する SMILES文字列（確信度順に最大25個）
* 採点・正解判定の基準: inchikey14（立体を除いた2D骨格の一致で判定）
* `train.parquet` に含まれる正解ラベル群:
* `normalized_smiles`（正解の構造式文字列）
* `inchikey14`（正解の2D骨格ハッシュ・評価キー）
* `inchikey`（正解の完全なInChIKey）
* `molecular_formula`（正解の中性分子式)

2. 説明変数（モデルの入力となる特徴量）
* 壊す前の測定値: `precursor_mz`
* 破片（フラグメント）データ: `ms2_mzs`, `ms2_normalized_intensities`, `base_peak_intensity`
* 実験・測定条件: `adduct`, `collision_energy_ev`, `ionization_mode`, `instrument_type`

3. その他（メタデータ・データ管理用）
* ID: `molecule_id`, `spectrum_id`（testのみ）
* 出所・品質・元表記: `ingest_lib`, `precursor_error_ppm`, `adduct_orig`, `collision_energy_orig`, `collision_energy_orig_units`, `num_peaks`（trainのみ等）
